# Connect to google account

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


# Global Variables

In [2]:
import easydict
args = easydict.EasyDict()

# file path
args.default_path = "/content/drive/MyDrive/Study/kaggle_competition/data/" # 파일이 존재하는 폴더
args.train_csv = args.default_path + "train.csv" # 학습용 데이터셋
args.test_csv = args.default_path + "test.csv" # 제출용 예측 데이터셋
args.default_submission_csv = args.default_path + "submission.csv" # 제출에 사용할 파일
args.submission_csv = "model_with_cv_0820_trial01.csv" # 제출용 파일


In [3]:
args.train_csv == "/content/drive/MyDrive/Study/kaggle_competition/data/train.csv"

True

# reset_seeds

In [4]:
import os
import numpy as np
import random
import torch

def reset_seeds(seed=42):
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)    # 파이썬 환경변수 시드 고정
  np.random.seed(seed)
  torch.manual_seed(seed) # cpu 연산 무작위 고정
  torch.cuda.manual_seed(seed) # gpu 연산 무작위 고정
  torch.backends.cudnn.deterministic = True  # cuda 라이브러리에서 Deterministic(결정론적)으로 예측하기 (예측에 대한 불확실성 제거 )

# Load Dataset

In [5]:
import numpy as np
import pandas as pd

In [6]:
ori_train = pd.read_csv(args.train_csv)
ori_test = pd.read_csv(args.test_csv)

ori_train.shape, ori_test.shape

((916, 12), (393, 11))

# EDA

In [7]:
ori_train.head()

,passengerid,survived,pclass,name,gender,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,0,2,"Wheeler, Mr. Edwin Frederick""""",male,NaN,0,0,SC/PARIS 2159,12.8750,NaN,S
1,1,0,3,"Henry, Miss. Delia",female,NaN,0,0,382649,7.7500,NaN,Q
2,2,1,1,"Hays, Mrs. Charles Melville (Clara Jennings Gr...",female,52.0,1,1,12749,93.5000,B69,S
3,3,1,3,"Andersson, Mr. August Edvard (""Wennerstrom"")",male,27.0,0,0,350043,7.7958,NaN,S
4,4,0,2,"Hold, Mr. Stephen",male,44.0,1,0,26707,26.0000,NaN,S


# Cleaning Dataset

In [8]:
ori_train.shape, ori_test.shape

((916, 12), (393, 11))

In [9]:
# 필요없는 컬럼 제거
ori_train.drop(['passengerid'], axis=1, inplace=True)
ori_test.drop(['passengerid'], axis=1, inplace=True)

ori_train.shape, ori_test.shape

((916, 11), (393, 10))

In [10]:
# drop -> 삭제하다.
# duplicates -> row 데이터에 대한 중복데이터
# drop_duplicates -> row 데이터들 중 중복 데이터 삭제
ori_train.drop_duplicates(inplace=True, keep='first')
ori_test.drop_duplicates(inplace=True, keep='first')

ori_train.shape, ori_test.shape

((916, 11), (393, 10))

## 결측치 제거

In [11]:
(ori_train.isnull().sum() / len(ori_train)).sort_values(ascending=False)

,0
cabin,0.783843
age,0.196507
embarked,0.001092
name,0.000000
pclass,0.000000
survived,0.000000
gender,0.000000
parch,0.000000
sibsp,0.000000
fare,0.000000


In [27]:
ori_train_none_cols = ori_train.isnull().sum()[ori_train.isnull().sum() > 0].index
ori_train_none_cols

Index(['age', 'cabin', 'embarked'], dtype='object')

In [28]:
ori_test_none_cols = ori_test.isnull().sum()[ori_test.isnull().sum() > 0].index
ori_test_none_cols

Index(['age', 'fare', 'cabin', 'embarked'], dtype='object')

In [34]:
none_cols = list(set(ori_train_none_cols) | set(ori_test_none_cols))
none_cols # 결측치 컬럼

['cabin', 'age', 'fare', 'embarked']

In [35]:
for col in none_cols:
  try:
    # 통계 값 추출
    _value = ori_train[col].mean()
  except:
    _value = ori_train[col].mode()[0]
  finally:
    # 결측치에 통계값 넣기
    ori_train[col].fillna(_value, inplace=True)
    ori_test[col].fillna(_value, inplace=True)

/tmp/ipython-input-1678781946.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ori_train[col].fillna(_value, inplace=True)
/tmp/ipython-input-1678781946.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

In [36]:
ori_train.isnull().sum().sum(), ori_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

# 인코딩 처리

In [37]:
ori_train.columns

Index(['survived', 'pclass', 'name', 'gender', 'age', 'sibsp', 'parch',
       'ticket', 'fare', 'cabin', 'embarked'],
      dtype='object')

In [ ]:
drop_cols = [
    'name', 'ticket'
]
ori_train.drop(drop_cols, axis=1, inplace=True)
ori_test.drop(drop_cols, axis=1, inplace=True)

In [40]:
encoding_cols = [
    'pclass', 'gender', 'cabin', 'embarked'
]

for col in encoding_cols:
  ori_train[col] = ori_train[col].astype('category') #.cat.codes
  ori_test[col] = ori_test[col].astype('category') #.cat.codes

In [41]:
ori_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   survived  916 non-null    int64   
 1   pclass    916 non-null    category
 2   gender    916 non-null    category
 3   age       916 non-null    float64 
 4   sibsp     916 non-null    int64   
 5   parch     916 non-null    int64   
 6   fare      916 non-null    float64 
 7   cabin     916 non-null    category
 8   embarked  916 non-null    category
dtypes: category(4), float64(2), int64(3)
memory usage: 46.0 KB


# 모델링

In [42]:
ori_train.isnull().sum().sum(), ori_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

In [43]:
ori_train.shape, ori_test.shape

((916, 9), (393, 8))

In [44]:
train_x = ori_train.drop(['survived'], axis=1)
train_y = ori_train['survived']

train_x.shape, train_y.shape

((916, 8), (916,))

## 모델 생성

In [45]:
from lightgbm import LGBMClassifier, plot_importance

In [46]:
reset_seeds()

model = LGBMClassifier(verbose=-1)

## CV

In [47]:
from sklearn.model_selection import StratifiedKFold

reset_seeds()

cv = StratifiedKFold(n_splits=5, shuffle=True)

In [48]:
from sklearn.metrics import roc_auc_score
reset_seeds()

for i, (train_index, valid_index) in enumerate(cv.split(train_x, train_y)):
  # 학습용 데이터 -> features, targets
  tr_features, tr_targets = train_x.iloc[train_index], train_y.iloc[train_index]
  # 평가용 데이터 -> features, targests
  te_features, te_targets = train_x.iloc[valid_index], train_y.iloc[valid_index]

  # 모델 학습
  model.fit(tr_features, tr_targets)

  # 평가
  predictions = model.predict_proba(te_features)[:, 1]
  score = roc_auc_score(te_targets, predictions)
  print(f"{i+1}번째 점수는 {score}")

1번째 점수는 0.8636591478696743
2번째 점수는 0.8992499364352912
3번째 점수는 0.8617467581998475
4번째 점수는 0.8789727943046021
5번째 점수는 0.8966437833714721


# 제출용 만들자

In [49]:
reset_seeds()
ori_pred = model.predict_proba(ori_test)

len(ori_pred) == len(ori_test)

True